# 02 - Calidad y limpieza de datos

En este notebook se analizan los problemas de calidad detectados durante la inspección inicial y se aplican decisiones de limpieza justificadas.

El objetivo es preparar una versión limpia del dataset sin modificar el archivo original ubicado en `data/raw`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/PI_Mineria_Datos_1/notebooks

/content/drive/MyDrive/PI_Mineria_Datos_1/notebooks


In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
# Cargamos el dataset original desde la carpeta raw

df_raw = pd.read_json("../data/raw/streaming_users_dirty.json")

df_raw.head()

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


In [ ]:
# Creamos una copia para trabajar sin modificar el dataset original

df = df_raw.copy()

In [ ]:
# Guardamos la cantidad inicial de filas para medir el impacto de la limpieza

filas_iniciales = df.shape[0]

print("Filas iniciales:", filas_iniciales)
print("Columnas iniciales:", df.shape[1])

Filas iniciales: 8160
Columnas iniciales: 8


In [ ]:
# Lista donde vamos a registrar cada paso de limpieza

log_limpieza = []

In [ ]:
# Función para registrar cada transformación aplicada al dataset

def registrar_paso(paso, descripcion):
    log_limpieza.append({
        "Paso": paso,
        "Descripción": descripcion,
        "Filas": df.shape[0],
        "Nulos": df.isnull().sum().sum(),
        "Retención (%)": round((df.shape[0] / filas_iniciales) * 100, 2)
    })

In [ ]:
# Registramos el estado inicial del dataset

registrar_paso(
    "Carga inicial",
    "Se carga el dataset original y se crea una copia de trabajo sin modificar el archivo raw."
)

In [ ]:
pd.DataFrame(log_limpieza)

,Paso,Descripción,Filas,Nulos,Retención (%)
0,Carga inicial,Se carga el dataset original y se crea una cop...,8160,753,100.0


In [ ]:
# Cantidad de filas completamente duplicadas

duplicados_completos = df.duplicated().sum()

print("Filas duplicadas completas:", duplicados_completos)

Filas duplicadas completas: 126


In [ ]:
# Visualizamos algunos registros duplicados completos

df[df.duplicated()].head()

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
8000,10156,43,Básico,592.8,Brasil,Romance,2021-10-25,0
8001,13860,22,Básico,834.8,Chile,Romance,2021-01-05,0
8002,17431,31,Estándar,1079.8,Chile,Documental,2023-04-12,0
8005,10375,32,Premium,1223.3,Uruguay,Comedia,2020-12-22,2
8006,11988,42,Estándar,1048.0,Uruguay,Drama,2023-01-12,0


### Tratamiento de duplicados completos

**Evidencia:** Se detectaron filas completamente duplicadas, es decir, registros repetidos en todas sus columnas.

**Decisión:** Se eliminan los duplicados completos porque no aportan información nueva al análisis y pueden alterar conteos, proporciones y visualizaciones.

**Impacto esperado:** Se reduce la cantidad de filas, pero no se pierde información única, ya que solo se eliminan copias exactas de registros ya existentes.

In [ ]:
# Eliminamos filas completamente duplicadas

df = df.drop_duplicates()

In [ ]:
registrar_paso(
    "Eliminación de duplicados completos",
    "Se eliminaron filas repetidas en todas sus columnas para evitar duplicación de información."
)

In [ ]:
pd.DataFrame(log_limpieza)

,Paso,Descripción,Filas,Nulos,Retención (%)
0,Carga inicial,Se carga el dataset original y se crea una cop...,8160,753,100.00
1,Eliminación de duplicados completos,Se eliminaron filas repetidas en todas sus col...,8034,753,98.46


In [ ]:
# Cantidad de user_id repetidos después de eliminar duplicados completos

user_id_repetidos = df["user_id"].duplicated().sum()

print("Cantidad de user_id repetidos:", user_id_repetidos)

Cantidad de user_id repetidos: 34


In [ ]:
# Visualizamos registros que comparten el mismo user_id

df[df["user_id"].duplicated(keep=False)].sort_values("user_id").head(20)

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
59,10059,39,Estándar,2976.6,Brasil,DRAMA,2024-06-22,1
8090,10059,39,Estándar,824.8,Brasil,Drama,2024-06-22,1
8126,10721,32,Estándar,959.0,Colombia,Documental,2021-09-02,2
721,10721,32,Estándar,959.0,Colombia,Documental,None,2
797,10797,31,Básico,-1.0,México,Comedia,2023-07-01,1
8018,10797,31,Básico,410.4,México,Comedia,2023-07-01,1
8076,11092,31,Estándar,959.6,Chile,Romance,2025-01-05,2
1092,11092,31,Estándar,959.6,Chile,Romance,2025-01-05,2
1222,11222,13,Estándar,1321.8,MEX,Documental,2019-02-08,0
8157,11222,13,Estándar,1321.8,México,Documental,2019-02-08,0


In [ ]:
# Cantidad de apariciones por user_id

conteo_user_id = df["user_id"].value_counts()

conteo_user_id[conteo_user_id > 1].head(20)

,count
user_id,
11292,2
17520,2
10721,2
15704,2
14841,2
13293,2
16045,2
10797,2
11222,2


### Revisión de `user_id` repetidos

**Evidencia:** Se detectan valores repetidos en la columna `user_id`, incluso después de eliminar duplicados completos.

Esto indica que existen usuarios que aparecen más de una vez en el dataset, pero no necesariamente con filas idénticas.

**Decisión:** No se eliminan automáticamente todos los registros con `user_id` repetido. Primero se analiza si corresponden a registros duplicados reales o si contienen diferencias en alguna variable.

**Justificación:** Como `user_id` debería identificar de manera única a cada usuario, su repetición representa un problema de calidad. Sin embargo, eliminar registros sin revisar podría provocar pérdida de información.

In [ ]:
# Guardamos todos los registros que tienen user_id repetido

ids_repetidos = conteo_user_id[conteo_user_id > 1].index

df_user_id_repetidos = df[df["user_id"].isin(ids_repetidos)].sort_values("user_id")

df_user_id_repetidos.head(20)

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
59,10059,39,Estándar,2976.6,Brasil,DRAMA,2024-06-22,1
8090,10059,39,Estándar,824.8,Brasil,Drama,2024-06-22,1
8126,10721,32,Estándar,959.0,Colombia,Documental,2021-09-02,2
721,10721,32,Estándar,959.0,Colombia,Documental,None,2
797,10797,31,Básico,-1.0,México,Comedia,2023-07-01,1
8018,10797,31,Básico,410.4,México,Comedia,2023-07-01,1
8076,11092,31,Estándar,959.6,Chile,Romance,2025-01-05,2
1092,11092,31,Estándar,959.6,Chile,Romance,2025-01-05,2
1222,11222,13,Estándar,1321.8,MEX,Documental,2019-02-08,0
8157,11222,13,Estándar,1321.8,México,Documental,2019-02-08,0


In [ ]:
# Analizamos cuántos valores distintos hay en cada columna para cada user_id repetido

resumen_user_id_repetidos = df_user_id_repetidos.groupby("user_id").agg(
    registros=("user_id", "size"),
    edades_distintas=("age", lambda x: x.nunique(dropna=False)),
    planes_distintos=("subscription_plan", lambda x: x.nunique(dropna=False)),
    tiempos_distintos=("monthly_watch_time_mins", lambda x: x.nunique(dropna=False)),
    paises_distintos=("country", lambda x: x.nunique(dropna=False)),
    generos_distintos=("favorite_genre", lambda x: x.nunique(dropna=False)),
    fechas_distintas=("last_login_date", lambda x: x.nunique(dropna=False)),
    tickets_distintos=("customer_support_tickets", lambda x: x.nunique(dropna=False))
)

resumen_user_id_repetidos.head(20)

,registros,edades_distintas,planes_distintos,tiempos_distintos,paises_distintos,generos_distintos,fechas_distintas,tickets_distintos
user_id,,,,,,,,
10059,2,1,1,2,1,2,1,1
10721,2,1,1,1,1,1,2,1
10797,2,1,1,2,1,1,1,1
11092,2,1,1,1,1,2,1,1
11222,2,1,1,1,2,1,1,1
11292,2,1,1,1,2,1,1,1
11586,2,1,1,1,1,2,1,1
11746,2,1,1,1,1,2,1,1
11796,2,1,1,1,1,1,2,1


In [ ]:
# Contamos en cuántas columnas hay diferencias para cada user_id repetido

columnas_diferencias = [
    "edades_distintas",
    "planes_distintos",
    "tiempos_distintos",
    "paises_distintos",
    "generos_distintos",
    "fechas_distintas",
    "tickets_distintos"
]

resumen_user_id_repetidos["columnas_con_diferencias"] = (
    resumen_user_id_repetidos[columnas_diferencias] > 1
).sum(axis=1)

resumen_user_id_repetidos.sort_values(
    "columnas_con_diferencias",
    ascending=False
).head(20)

,registros,edades_distintas,planes_distintos,tiempos_distintos,paises_distintos,generos_distintos,fechas_distintas,tickets_distintos,columnas_con_diferencias
user_id,,,,,,,,,
10059,2,1,1,2,1,2,1,1,2
13293,2,1,1,1,1,2,2,1,2
10721,2,1,1,1,1,1,2,1,1
10797,2,1,1,2,1,1,1,1,1
11222,2,1,1,1,2,1,1,1,1
11092,2,1,1,1,1,2,1,1,1
11586,2,1,1,1,1,2,1,1,1
11746,2,1,1,1,1,2,1,1,1
11796,2,1,1,1,1,1,2,1,1


### Análisis adicional de `user_id` repetidos

Al revisar los registros con `user_id` repetido, se observa que no todos son duplicados completos.  
En varios casos, el mismo usuario presenta diferencias en variables como `monthly_watch_time_mins`, `country`, `favorite_genre`, `last_login_date` o `age`.

Por este motivo, no se eliminan automáticamente en esta etapa.

La decisión es conservar temporalmente estos registros y posponer la consolidación final hasta después de normalizar categorías, validar fechas y tratar valores inválidos. Esto evita eliminar información útil o conservar una versión incorrecta del registro.

In [ ]:
registrar_paso(
    "Revisión de user_id repetidos",
    "Se detectaron user_id repetidos con diferencias entre registros. No se eliminan en esta etapa; se pospone su consolidación hasta finalizar la normalización y validación de datos."
)

In [ ]:
pd.DataFrame(log_limpieza)

,Paso,Descripción,Filas,Nulos,Retención (%)
0,Carga inicial,Se carga el dataset original y se crea una cop...,8160,753,100.00
1,Eliminación de duplicados completos,Se eliminaron filas repetidas en todas sus col...,8034,753,98.46
2,Revisión de user_id repetidos,Se detectaron user_id repetidos con diferencia...,8034,753,98.46


In [ ]:
# Revisamos las categorías originales de subscription_plan

df["subscription_plan"].value_counts(dropna=False)

,count
subscription_plan,
Básico,3395
Estándar,2669
Premium,1490
basico,60
BASICO,52
Basic,52
básico,50
Std,48
Estándar,46


### Normalización de `subscription_plan`

**Evidencia:** La variable `subscription_plan` presenta categorías que representan el mismo plan, pero escritas de diferentes formas, por ejemplo con mayúsculas, minúsculas, acentos, idioma inglés o errores de escritura.

**Decisión:** Se normalizan las categorías para unificar los planes en tres valores principales: `Básico`, `Estándar` y `Premium`.

**Justificación:** Si no se corrige esta inconsistencia, el análisis podría contar como planes diferentes a valores que en realidad representan la misma categoría.

**Impacto esperado:** No se eliminan filas. Solo se mejora la consistencia de la variable categórica.

In [ ]:
# Normalizamos la variable subscription_plan

df["subscription_plan"] = (
    df["subscription_plan"]
    .astype("string")
    .str.strip()
    .str.lower()
)

mapa_planes = {
    "basico": "Básico",
    "básico": "Básico",
    "basic": "Básico",

    "estandar": "Estándar",
    "estándar": "Estándar",
    "std": "Estándar",
    "standard": "Estándar",

    "premium": "Premium",
    "premiun": "Premium"
}

df["subscription_plan"] = df["subscription_plan"].replace(mapa_planes)

In [ ]:
# Revisamos las categorías luego de la normalización

df["subscription_plan"].value_counts(dropna=False)

,count
subscription_plan,
Básico,3609
Estándar,2833
Premium,1592


In [ ]:
registrar_paso(
    "Normalización de subscription_plan",
    "Se unificaron variantes de escritura de los planes de suscripción en Básico, Estándar y Premium."
)

In [ ]:
pd.DataFrame(log_limpieza)

,Paso,Descripción,Filas,Nulos,Retención (%)
0,Carga inicial,Se carga el dataset original y se crea una cop...,8160,753,100.00
1,Eliminación de duplicados completos,Se eliminaron filas repetidas en todas sus col...,8034,753,98.46
2,Revisión de user_id repetidos,Se detectaron user_id repetidos con diferencia...,8034,753,98.46
3,Normalización de subscription_plan,Se unificaron variantes de escritura de los pl...,8034,753,98.46


In [ ]:
# Revisamos las categorías originales de country

df["country"].value_counts(dropna=False)

,count
country,
Chile,1118
Brasil,1115
México,1106
Uruguay,1105
Colombia,1099
Perú,1096
Argentina,1075
colombia,27
méxico,25


### Normalización de `country`

**Evidencia:** La variable `country` presenta países escritos de diferentes formas, incluyendo mayúsculas, minúsculas, abreviaturas y nombres en distintos idiomas.

**Decisión:** Se normalizan los valores para unificar las distintas variantes bajo un mismo nombre de país.

**Justificación:** Si no se corrige esta inconsistencia, el análisis podría contar como países diferentes a valores que representan el mismo lugar.

**Impacto esperado:** No se eliminan filas. Solo se mejora la consistencia de la variable categórica.

In [ ]:
# Normalizamos la variable country

df["country"] = (
    df["country"]
    .astype("string")
    .str.strip()
    .str.lower()
)

mapa_paises = {
    "argentina": "Argentina",
    "arg": "Argentina",

    "brasil": "Brasil",
    "brazil": "Brasil",
    "bra": "Brasil",

    "mexico": "México",
    "méxico": "México",
    "mex": "México",

    "chile": "Chile",
    "chl": "Chile",

    "uruguay": "Uruguay",
    "ury": "Uruguay",

    "colombia": "Colombia",
    "col": "Colombia",

    "peru": "Perú",
    "perú": "Perú",
    "per": "Perú"
}

df["country"] = df["country"].replace(mapa_paises)

In [ ]:
# Revisamos las categorías luego de la normalización

df["country"].value_counts(dropna=False)

,count
country,
Chile,1167
Brasil,1164
México,1162
Uruguay,1146
Colombia,1145
Perú,1139
Argentina,1111


In [ ]:
registrar_paso(
    "Normalización de country",
    "Se unificaron variantes de escritura, abreviaturas e idiomas en la variable country."
)

In [ ]:
pd.DataFrame(log_limpieza)

,Paso,Descripción,Filas,Nulos,Retención (%)
0,Carga inicial,Se carga el dataset original y se crea una cop...,8160,753,100.00
1,Eliminación de duplicados completos,Se eliminaron filas repetidas en todas sus col...,8034,753,98.46
2,Revisión de user_id repetidos,Se detectaron user_id repetidos con diferencia...,8034,753,98.46
3,Normalización de subscription_plan,Se unificaron variantes de escritura de los pl...,8034,753,98.46
4,Normalización de country,"Se unificaron variantes de escritura, abreviat...",8034,753,98.46


In [ ]:
# Revisamos las categorías originales de favorite_genre

df["favorite_genre"].value_counts(dropna=False)

,count
favorite_genre,
Comedia,1095
Drama,1088
Thriller,1077
Documental,1070
Romance,1070
Acción,1066
Crime,1049
None,240
Action,22


### Normalización de `favorite_genre`

**Evidencia:** La variable `favorite_genre` presenta géneros escritos de diferentes formas, incluyendo mayúsculas, minúsculas, acentos, abreviaturas y nombres en inglés.

**Decisión:** Se normalizan los valores para unificar las distintas variantes bajo una misma categoría.

**Justificación:** Si no se corrige esta inconsistencia, el análisis podría contar como géneros distintos a valores que representan la misma preferencia de contenido.

**Impacto esperado:** No se eliminan filas. Solo se mejora la consistencia de la variable categórica.

In [ ]:
# Normalizamos la variable favorite_genre

df["favorite_genre"] = (
    df["favorite_genre"]
    .astype("string")
    .str.strip()
    .str.lower()
)

mapa_generos = {
    "accion": "Acción",
    "acción": "Acción",
    "action": "Acción",

    "comedia": "Comedia",
    "comedy": "Comedia",

    "documental": "Documental",
    "documentary": "Documental",
    "doc": "Documental",

    "drama": "Drama",

    "romance": "Romance",

    "crime": "Crimen",
    "crimen": "Crimen",

    "thriller": "Thriller",
    "thriler": "Thriller",

    "none": pd.NA,
    "not_available": pd.NA
}

df["favorite_genre"] = df["favorite_genre"].replace(mapa_generos)

In [ ]:
# Revisamos las categorías luego de la normalización

df["favorite_genre"].value_counts(dropna=False)

,count
favorite_genre,
Comedia,1141
Drama,1121
Romance,1113
Documental,1111
Acción,1110
Thriller,1109
Crimen,1089
<NA>,240


In [ ]:
registrar_paso(
    "Normalización de favorite_genre",
    "Se unificaron variantes de escritura, abreviaturas e idiomas en la variable favorite_genre."
)

In [ ]:
pd.DataFrame(log_limpieza)

,Paso,Descripción,Filas,Nulos,Retención (%)
0,Carga inicial,Se carga el dataset original y se crea una cop...,8160,753,100.00
1,Eliminación de duplicados completos,Se eliminaron filas repetidas en todas sus col...,8034,753,98.46
2,Revisión de user_id repetidos,Se detectaron user_id repetidos con diferencia...,8034,753,98.46
3,Normalización de subscription_plan,Se unificaron variantes de escritura de los pl...,8034,753,98.46
4,Normalización de country,"Se unificaron variantes de escritura, abreviat...",8034,753,98.46
5,Normalización de favorite_genre,"Se unificaron variantes de escritura, abreviat...",8034,753,98.46


In [ ]:
# Revisamos estadísticas de la variable age

df["age"].describe().round(2)

,age
count,8034.00
mean,34.10
std,14.56
min,-5.00
25%,25.00
50%,33.00
75%,42.00
max,150.00


In [ ]:
# Detectamos edades inválidas
# Consideramos inválidas las edades menores o iguales a 0 y mayores a 100

edades_invalidas = df[(df["age"] <= 0) | (df["age"] > 100)]

edades_invalidas[["user_id", "age"]].head(20)

,user_id,age
49,10049,0
194,10194,130
310,10310,0
324,10324,130
426,10426,150
442,10442,-5
529,10529,130
573,10573,-5
640,10640,150
655,10655,130


In [ ]:
# Cantidad de edades inválidas detectadas

len(edades_invalidas)

98

### Tratamiento de valores inválidos en `age`

**Evidencia:** Se detectaron edades imposibles o poco razonables, como valores menores o iguales a 0 y edades mayores a 100.

**Decisión:** Estos valores se consideran inválidos y se reemplazan por valores nulos temporales. Luego se imputan utilizando la mediana de las edades válidas.

**Justificación:** No se eliminan las filas porque pueden contener información útil en otras variables. Se utiliza la mediana porque es más robusta que la media frente a valores extremos.

**Impacto esperado:** Se corrigen edades imposibles sin reducir la cantidad de registros del dataset.

In [ ]:
# Reemplazamos edades inválidas por NaN

df.loc[(df["age"] <= 0) | (df["age"] > 100), "age"] = np.nan

In [ ]:
# Calculamos la mediana de age luego de marcar las edades inválidas como nulas

mediana_edad = df["age"].median()

print("Mediana de edad:", mediana_edad)

Mediana de edad: 33.0


In [ ]:
# Imputamos los valores nulos de age con la mediana

df["age"] = df["age"].fillna(mediana_edad)

# Dejamos age como entero
df["age"] = df["age"].round().astype(int)

In [ ]:
# Verificamos que ya no existan edades inválidas

df[(df["age"] <= 0) | (df["age"] > 100)]

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets


In [ ]:
registrar_paso(
    "Tratamiento de edades inválidas",
    "Se reemplazaron edades menores o iguales a 0 y mayores a 100 por la mediana de las edades válidas.")

In [ ]:
pd.DataFrame(log_limpieza)

,Paso,Descripción,Filas,Nulos,Retención (%)
0,Carga inicial,Se carga el dataset original y se crea una cop...,8160,753,100.00
1,Eliminación de duplicados completos,Se eliminaron filas repetidas en todas sus col...,8034,753,98.46
2,Revisión de user_id repetidos,Se detectaron user_id repetidos con diferencia...,8034,753,98.46
3,Normalización de subscription_plan,Se unificaron variantes de escritura de los pl...,8034,753,98.46
4,Normalización de country,"Se unificaron variantes de escritura, abreviat...",8034,753,98.46
5,Normalización de favorite_genre,"Se unificaron variantes de escritura, abreviat...",8034,753,98.46
6,Tratamiento de edades inválidas,Se reemplazaron edades menores o iguales a 0 y...,8034,753,98.46


In [ ]:
# Revisamos estadísticas de la variable monthly_watch_time_mins

df["monthly_watch_time_mins"].describe().round(2)

,monthly_watch_time_mins
count,7841.0
mean,1111.8
std,5352.6
min,-120.0
25%,488.0
50%,757.2
75%,1044.4
max,99999.0


In [ ]:
# Detectamos tiempos de visualización inválidos o imposibles

tiempos_invalidos = df[
    (df["monthly_watch_time_mins"] < 0) |
    (df["monthly_watch_time_mins"] > 43200)
]

tiempos_invalidos[["user_id", "monthly_watch_time_mins"]].head(20)

,user_id,monthly_watch_time_mins
176,10176,50000.0
430,10430,99999.0
443,10443,-120.0
481,10481,50000.0
588,10588,99999.0
797,10797,-1.0
1072,11072,99999.0
1126,11126,-1.0
1186,11186,-120.0
1250,11250,99999.0


In [ ]:
# Cantidad de tiempos inválidos o imposibles

len(tiempos_invalidos)

80

### Tratamiento de valores inválidos en `monthly_watch_time_mins`

**Evidencia:** Se detectaron valores negativos y valores superiores a la cantidad máxima posible de minutos en un mes.

**Decisión:** Los valores negativos y los valores mayores a 43.200 minutos se consideran inválidos. Se reemplazan por valores nulos temporales y luego se imputan con la mediana de los tiempos válidos.

**Justificación:** Un tiempo de visualización no puede ser negativo ni superar la cantidad total de minutos disponibles en un mes. Se utiliza la mediana porque es más robusta frente a valores extremos que la media.

**Impacto esperado:** Se corrigen valores imposibles sin eliminar filas completas del dataset.

In [ ]:
# Reemplazamos tiempos inválidos por NaN

df.loc[
    (df["monthly_watch_time_mins"] < 0) |
    (df["monthly_watch_time_mins"] > 43200),
    "monthly_watch_time_mins"
] = np.nan

In [ ]:
# Calculamos la mediana de los tiempos válidos

mediana_watch_time = df["monthly_watch_time_mins"].median()

print("Mediana de tiempo de visualización:", mediana_watch_time)

Mediana de tiempo de visualización: 758.5


In [ ]:
# Imputamos valores nulos de monthly_watch_time_mins con la mediana

df["monthly_watch_time_mins"] = df["monthly_watch_time_mins"].fillna(mediana_watch_time)

In [ ]:
# Verificamos que no queden tiempos negativos o imposibles

df[
    (df["monthly_watch_time_mins"] < 0) |
    (df["monthly_watch_time_mins"] > 43200)
]

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets


In [ ]:
registrar_paso(
    "Tratamiento de tiempos de visualización inválidos",
    "Se reemplazaron tiempos negativos y mayores a 43.200 minutos por la mediana de los tiempos válidos.")

In [ ]:
pd.DataFrame(log_limpieza)

,Paso,Descripción,Filas,Nulos,Retención (%)
0,Carga inicial,Se carga el dataset original y se crea una cop...,8160,753,100.00
1,Eliminación de duplicados completos,Se eliminaron filas repetidas en todas sus col...,8034,753,98.46
2,Revisión de user_id repetidos,Se detectaron user_id repetidos con diferencia...,8034,753,98.46
3,Normalización de subscription_plan,Se unificaron variantes de escritura de los pl...,8034,753,98.46
4,Normalización de country,"Se unificaron variantes de escritura, abreviat...",8034,753,98.46
5,Normalización de favorite_genre,"Se unificaron variantes de escritura, abreviat...",8034,753,98.46
6,Tratamiento de edades inválidas,Se reemplazaron edades menores o iguales a 0 y...,8034,753,98.46
7,Tratamiento de tiempos de visualización inválidos,Se reemplazaron tiempos negativos y mayores a ...,8034,560,98.46


In [ ]:
# Revisamos estadísticas de customer_support_tickets

df["customer_support_tickets"].describe().round(2)

,customer_support_tickets
count,8034.00
mean,1.82
std,11.42
min,-1.00
25%,0.00
50%,1.00
75%,1.00
max,150.00


In [ ]:
# Detectamos tickets negativos o extremadamente altos

tickets_invalidos = df[
    (df["customer_support_tickets"] < 0) |
    (df["customer_support_tickets"] > 20)
]

tickets_invalidos[["user_id", "customer_support_tickets"]].head(20)

,user_id,customer_support_tickets
0,10000,99
300,10300,150
366,10366,99
378,10378,-1
578,10578,150
609,10609,150
682,10682,-1
811,10811,150
1052,11052,150
1099,11099,-1


In [ ]:
# Cantidad de valores inválidos o sospechosos en customer_support_tickets

len(tickets_invalidos)

96

### Tratamiento de valores inválidos en `customer_support_tickets`

**Evidencia:** Se detectaron valores negativos y valores extremadamente altos en la variable `customer_support_tickets`.

**Decisión:** Los valores negativos y los valores mayores a 20 se consideran no confiables para el análisis. Se reemplazan por valores nulos temporales y luego se imputan con la mediana de los valores válidos.

**Justificación:** La cantidad de tickets de soporte no puede ser negativa. Además, valores demasiado altos pueden corresponder a errores de carga o casos atípicos extremos. Se utiliza la mediana porque es más robusta frente a valores extremos que la media.

**Impacto esperado:** Se corrigen valores inválidos o extremadamente sospechosos sin eliminar filas completas del dataset.

In [ ]:
# Reemplazamos tickets inválidos o sospechosos por NaN

df.loc[
    (df["customer_support_tickets"] < 0) |
    (df["customer_support_tickets"] > 20),
    "customer_support_tickets"
] = np.nan

In [ ]:
# Calculamos la mediana de tickets válidos

mediana_tickets = df["customer_support_tickets"].median()

print("Mediana de tickets de soporte:", mediana_tickets)

Mediana de tickets de soporte: 1.0


In [ ]:
# Imputamos los valores nulos de customer_support_tickets con la mediana

df["customer_support_tickets"] = df["customer_support_tickets"].fillna(mediana_tickets)

# Dejamos la variable como entero porque representa una cantidad
df["customer_support_tickets"] = df["customer_support_tickets"].round().astype(int)

In [ ]:
# Verificamos que no queden tickets negativos o mayores a 20

df[
    (df["customer_support_tickets"] < 0) |
    (df["customer_support_tickets"] > 20)
]

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets


In [ ]:
registrar_paso(
    "Tratamiento de tickets de soporte inválidos",
    "Se reemplazaron tickets negativos y mayores a 20 por la mediana de los valores válidos.")

In [ ]:
pd.DataFrame(log_limpieza)

,Paso,Descripción,Filas,Nulos,Retención (%)
0,Carga inicial,Se carga el dataset original y se crea una cop...,8160,753,100.00
1,Eliminación de duplicados completos,Se eliminaron filas repetidas en todas sus col...,8034,753,98.46
2,Revisión de user_id repetidos,Se detectaron user_id repetidos con diferencia...,8034,753,98.46
3,Normalización de subscription_plan,Se unificaron variantes de escritura de los pl...,8034,753,98.46
4,Normalización de country,"Se unificaron variantes de escritura, abreviat...",8034,753,98.46
5,Normalización de favorite_genre,"Se unificaron variantes de escritura, abreviat...",8034,753,98.46
6,Tratamiento de edades inválidas,Se reemplazaron edades menores o iguales a 0 y...,8034,753,98.46
7,Tratamiento de tiempos de visualización inválidos,Se reemplazaron tiempos negativos y mayores a ...,8034,560,98.46
8,Tratamiento de tickets de soporte inválidos,Se reemplazaron tickets negativos y mayores a ...,8034,560,98.46


In [ ]:
# Revisamos algunos valores originales de last_login_date

df["last_login_date"].value_counts(dropna=False).head(20)

,count
last_login_date,
None,320
2026-15-03,20
0000-00-00,17
2029-01-01,15
not_available,14
31-02-2022,13
2019-01-17,10
2024-08-29,9
2025-09-29,9


In [ ]:
# Convertimos last_login_date a formato fecha
# Los valores que no puedan convertirse quedarán como NaT

fechas_convertidas = pd.to_datetime(
    df["last_login_date"],
    errors="coerce",
    format="mixed"
)

fechas_convertidas.head()

,last_login_date
0,2025-03-04
1,2019-04-02
2,2018-04-13
3,2021-01-31
4,2020-09-30


In [ ]:
# Cantidad de fechas inválidas o faltantes luego de la conversión

fechas_invalidas = fechas_convertidas.isnull().sum()

print("Fechas inválidas o faltantes:", fechas_invalidas)

Fechas inválidas o faltantes: 384


In [ ]:
# Visualizamos ejemplos de fechas que no pudieron convertirse

df[fechas_convertidas.isnull()][["user_id", "last_login_date"]].head(20)

,user_id,last_login_date
76,10076,2026-15-03
95,10095,None
103,10103,None
115,10115,None
134,10134,None
138,10138,None
253,10253,31-02-2022
266,10266,None
275,10275,None
301,10301,2026-15-03


In [ ]:
# Definimos la fecha de referencia del análisis

fecha_referencia = pd.Timestamp("2026-06-28")

In [ ]:
# Detectamos fechas futuras respecto a la fecha de análisis

fechas_futuras = fechas_convertidas > fecha_referencia

df[fechas_futuras][["user_id", "last_login_date"]].head(20)

,user_id,last_login_date
58,10058,2029-01-01
69,10069,2029-01-01
162,10162,2029-01-01
737,10737,2029-01-01
908,10908,2029-01-01
1011,11011,2029-01-01
1580,11580,2029-01-01
1712,11712,2029-01-01
2272,12272,2029-01-01
2973,12973,2029-01-01


In [ ]:
# Cantidad de fechas futuras detectadas

fechas_futuras.sum()

np.int64(15)

### Tratamiento de `last_login_date`

**Evidencia:** La variable `last_login_date` contiene fechas con formatos mezclados, valores faltantes, valores no convertibles y fechas futuras.

**Decisión:** Se convierte la variable a formato fecha. Los valores que no puedan convertirse y las fechas futuras se marcan como `NaT`.

**Justificación:** Como `last_login_date` representa la fecha del último inicio de sesión, no corresponde inventar una fecha mediante imputación. Una fecha inválida o futura no es confiable para el análisis temporal.

**Impacto esperado:** Se mejora el tipo de dato de la variable y se identifican claramente los registros sin una fecha válida.

In [ ]:
# Reemplazamos la columna original por la versión convertida a fecha

df["last_login_date"] = fechas_convertidas

In [ ]:
# Marcamos como NaT las fechas futuras

df.loc[df["last_login_date"] > fecha_referencia, "last_login_date"] = pd.NaT

In [ ]:
# Verificamos el tipo de dato luego de la conversión

df["last_login_date"].dtype

dtype('<M8[ns]')

In [ ]:
# Verificamos que no queden fechas futuras

df[df["last_login_date"] > fecha_referencia]

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets


In [ ]:
registrar_paso(
    "Tratamiento de last_login_date",
    "Se convirtió last_login_date a formato fecha. Los valores inválidos, faltantes o futuros se marcaron como NaT.")

In [ ]:
pd.DataFrame(log_limpieza)

,Paso,Descripción,Filas,Nulos,Retención (%)
0,Carga inicial,Se carga el dataset original y se crea una cop...,8160,753,100.00
1,Eliminación de duplicados completos,Se eliminaron filas repetidas en todas sus col...,8034,753,98.46
2,Revisión de user_id repetidos,Se detectaron user_id repetidos con diferencia...,8034,753,98.46
3,Normalización de subscription_plan,Se unificaron variantes de escritura de los pl...,8034,753,98.46
4,Normalización de country,"Se unificaron variantes de escritura, abreviat...",8034,753,98.46
5,Normalización de favorite_genre,"Se unificaron variantes de escritura, abreviat...",8034,753,98.46
6,Tratamiento de edades inválidas,Se reemplazaron edades menores o iguales a 0 y...,8034,753,98.46
7,Tratamiento de tiempos de visualización inválidos,Se reemplazaron tiempos negativos y mayores a ...,8034,560,98.46
8,Tratamiento de tickets de soporte inválidos,Se reemplazaron tickets negativos y mayores a ...,8034,560,98.46
9,Tratamiento de last_login_date,Se convirtió last_login_date a formato fecha. ...,8034,639,98.46


In [ ]:
# Revisamos cuántos user_id repetidos quedan después de la limpieza previa

df["user_id"].duplicated().sum()

np.int64(34)

In [ ]:
# Visualizamos algunos user_id repetidos antes de consolidar

df[df["user_id"].duplicated(keep=False)].sort_values("user_id").head(20)

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
59,10059,39,Estándar,2976.6,Brasil,Drama,2024-06-22,1
8090,10059,39,Estándar,824.8,Brasil,Drama,2024-06-22,1
8126,10721,32,Estándar,959.0,Colombia,Documental,2021-09-02,2
721,10721,32,Estándar,959.0,Colombia,Documental,NaT,2
797,10797,31,Básico,758.5,México,Comedia,2023-07-01,1
8018,10797,31,Básico,410.4,México,Comedia,2023-07-01,1
8076,11092,31,Estándar,959.6,Chile,Romance,2025-01-05,2
1092,11092,31,Estándar,959.6,Chile,Romance,2025-01-05,2
1222,11222,13,Estándar,1321.8,México,Documental,2019-02-08,0
8157,11222,13,Estándar,1321.8,México,Documental,2019-02-08,0


### Consolidación de `user_id` repetidos

**Evidencia:** Luego de la limpieza previa, todavía existen registros con `user_id` repetido. Esto indica que algunos usuarios aparecen más de una vez en el dataset.

**Decisión:** Se conserva un único registro por `user_id`, priorizando el registro con mayor cantidad de datos válidos. En caso de empate, se conserva el registro con la fecha de último login más reciente.

**Justificación:** Como `user_id` debería identificar de manera única a cada usuario, mantener registros repetidos podría distorsionar conteos, proporciones y análisis por usuario. Sin embargo, no se elimina al azar: se elige el registro más completo y, si hay empate, el más reciente.

**Impacto esperado:** Se reduce la cantidad de filas, pero se conserva una versión más confiable de cada usuario.

In [ ]:
# Calculamos cuántos datos válidos tiene cada fila
# Esto nos ayuda a priorizar registros más completos

df["_campos_validos"] = df.notnull().sum(axis=1)

In [ ]:
# Ordenamos por user_id, cantidad de campos válidos y fecha más reciente

df = df.sort_values(
    by=["user_id", "_campos_validos", "last_login_date"],
    ascending=[True, False, False]
)

In [ ]:
# Conservamos un solo registro por user_id

df = df.drop_duplicates(subset="user_id", keep="first")

In [ ]:
# Eliminamos la columna auxiliar porque solo se usó para tomar la decisión

df = df.drop(columns=["_campos_validos"])

In [ ]:
# Verificamos que user_id sea único

df["user_id"].duplicated().sum()

np.int64(0)

In [ ]:
registrar_paso(
    "Consolidación de user_id repetidos",
    "Se conservó un único registro por user_id, priorizando registros más completos y, en caso de empate, la fecha de último login más reciente.")

In [ ]:
pd.DataFrame(log_limpieza)

,Paso,Descripción,Filas,Nulos,Retención (%)
0,Carga inicial,Se carga el dataset original y se crea una cop...,8160,753,100.00
1,Eliminación de duplicados completos,Se eliminaron filas repetidas en todas sus col...,8034,753,98.46
2,Revisión de user_id repetidos,Se detectaron user_id repetidos con diferencia...,8034,753,98.46
3,Normalización de subscription_plan,Se unificaron variantes de escritura de los pl...,8034,753,98.46
4,Normalización de country,"Se unificaron variantes de escritura, abreviat...",8034,753,98.46
5,Normalización de favorite_genre,"Se unificaron variantes de escritura, abreviat...",8034,753,98.46
6,Tratamiento de edades inválidas,Se reemplazaron edades menores o iguales a 0 y...,8034,753,98.46
7,Tratamiento de tiempos de visualización inválidos,Se reemplazaron tiempos negativos y mayores a ...,8034,560,98.46
8,Tratamiento de tickets de soporte inválidos,Se reemplazaron tickets negativos y mayores a ...,8034,560,98.46
9,Tratamiento de last_login_date,Se convirtió last_login_date a formato fecha. ...,8034,639,98.46


In [ ]:
# Revisamos dimensiones finales del dataset luego de la limpieza

print("Filas finales:", df.shape[0])
print("Columnas finales:", df.shape[1])

Filas finales: 8000
Columnas finales: 8


In [ ]:
# Revisamos valores nulos luego de la limpieza

df.isnull().sum()

,0
user_id,0
age,0
subscription_plan,0
monthly_watch_time_mins,0
country,0
favorite_genre,237
last_login_date,394
customer_support_tickets,0


In [ ]:
# Revisamos los tipos de datos finales

df.dtypes

,0
user_id,int64
age,int64
subscription_plan,string[python]
monthly_watch_time_mins,float64
country,string[python]
favorite_genre,string[python]
last_login_date,datetime64[ns]
customer_support_tickets,int64


In [ ]:
# Verificamos que no queden duplicados completos

df.duplicated().sum()

np.int64(0)

In [ ]:
# Verificamos que no queden user_id repetidos

df["user_id"].duplicated().sum()

np.int64(0)

In [ ]:
# Verificamos que no queden edades inválidas

df[(df["age"] <= 0) | (df["age"] > 100)]

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets


In [ ]:
# Verificamos que no queden tiempos negativos o imposibles

df[
    (df["monthly_watch_time_mins"] < 0) |
    (df["monthly_watch_time_mins"] > 43200)
]

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets


In [ ]:
# Verificamos que no queden tickets negativos o extremadamente altos

df[
    (df["customer_support_tickets"] < 0) |
    (df["customer_support_tickets"] > 20)
]

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets


### Revisión final del dataset limpio

Luego de aplicar las decisiones de limpieza, se verifica el estado final del dataset.

Se revisan dimensiones, valores nulos, tipos de datos, duplicados completos, `user_id` repetidos y valores inválidos en variables numéricas.

Esta verificación permite confirmar que las principales inconsistencias detectadas en la inspección inicial fueron tratadas antes de guardar la versión procesada del dataset.

In [ ]:
# Creamos la carpeta processed si todavía no existe

os.makedirs("../data/processed", exist_ok=True)

In [ ]:
# Guardamos el dataset limpio en la carpeta processed
df.to_csv("../data/processed/streaming_users_clean.csv", index=False)

print("Dataset limpio guardado correctamente en ../data/processed/streaming_users_clean.csv")

Dataset limpio guardado correctamente en ../data/processed/streaming_users_clean.csv


In [ ]:
# Convertimos el registro de limpieza en un DataFrame

pipeline_log = pd.DataFrame(log_limpieza)

pipeline_log

,Paso,Descripción,Filas,Nulos,Retención (%)
0,Carga inicial,Se carga el dataset original y se crea una cop...,8160,753,100.00
1,Eliminación de duplicados completos,Se eliminaron filas repetidas en todas sus col...,8034,753,98.46
2,Revisión de user_id repetidos,Se detectaron user_id repetidos con diferencia...,8034,753,98.46
3,Normalización de subscription_plan,Se unificaron variantes de escritura de los pl...,8034,753,98.46
4,Normalización de country,"Se unificaron variantes de escritura, abreviat...",8034,753,98.46
5,Normalización de favorite_genre,"Se unificaron variantes de escritura, abreviat...",8034,753,98.46
6,Tratamiento de edades inválidas,Se reemplazaron edades menores o iguales a 0 y...,8034,753,98.46
7,Tratamiento de tiempos de visualización inválidos,Se reemplazaron tiempos negativos y mayores a ...,8034,560,98.46
8,Tratamiento de tickets de soporte inválidos,Se reemplazaron tickets negativos y mayores a ...,8034,560,98.46
9,Tratamiento de last_login_date,Se convirtió last_login_date a formato fecha. ...,8034,639,98.46


In [ ]:
# Guardamos el log del pipeline en la carpeta logs

pipeline_log.to_csv("../logs/pipeline_log.csv", index=False)

print("Log del pipeline guardado correctamente en ../logs/pipeline_log.csv")

Log del pipeline guardado correctamente en ../logs/pipeline_log.csv


In [ ]:
# Verificamos que los archivos se hayan guardado correctamente

print("Archivos en data/processed:")
print(os.listdir("../data/processed"))

print("\nArchivos en logs:")
print(os.listdir("../logs"))

Archivos en data/processed:
['streaming_users_clean.csv']

Archivos en logs:
['resumen_inspeccion_inicial.csv', 'pipeline_log.csv']


## Cierre del notebook

En este notebook se realizó la limpieza y preparación del dataset a partir de los problemas detectados en la inspección inicial.

Se trataron duplicados completos, inconsistencias en variables categóricas, valores inválidos en variables numéricas, fechas no válidas y registros repetidos por `user_id`.

El dataset original ubicado en `data/raw` no fue modificado.  
La versión limpia fue guardada en `data/processed/streaming_users_clean.csv`.

Además, se generó el archivo `logs/pipeline_log.csv`, donde se documentan los pasos realizados, la cantidad de filas, los valores nulos y la retención de datos luego de cada transformación.